In [5]:
import time
import pandas as pd
from pytrends.exceptions import ResponseError

def safe_interest(pytrends, kw_list, timeframe, geo='US', max_retries=3):
    for attempt in range(max_retries):
        try:
            pytrends.build_payload(kw_list, timeframe=timeframe, geo=geo)
            return pytrends.interest_over_time()
        except Exception as e:
            wait = 60 * (attempt + 1)
            print(f"retry {attempt+1} after {wait}s: {e}")
            time.sleep(wait)
    return pd.DataFrame()


# Pytrends: Driver vs. Sponsor Search Interest

Goal: pull Google Trends interest for each driver and their primary sponsor, then check
whether sponsor search interest actually follows driver search interest (lead-lag),
rather than just moving together for unrelated reasons.

`safe_interest` above is the rate-limit-safe wrapper we'll use for every call.

In [2]:
from pytrends.request import TrendReq

pytrends = TrendReq(hl='en-US', tz=360)  # tz=360 = US Central

## Driver → sponsor pairs

`driver_stats_2025-6.csv` has a `Primary_Sponsor` column, but most rows are
`Other/Unknown` (only a handful of drivers have a resolved sponsor). Start from the
rows that *do* have a real sponsor, and add manual overrides for anyone important
that's missing (co-sponsors, sponsors that changed mid-season, etc).

In [4]:
driver_stats = pd.read_csv('data/processed/driver_stats_2025-6.csv')

known = driver_stats[driver_stats['Primary_Sponsor'] != 'Other/Unknown']
driver_sponsor_pairs = dict(zip(known['Driver'], known['Primary_Sponsor']))

# manual overrides / additions for drivers you care about that came back Other/Unknown
driver_sponsor_pairs.update({
    'Denny Hamlin': 'Progressive',
})

driver_sponsor_pairs

{'Austin Hill': "cheddar's Scratch Kitchen",
 'Brad Keselowski': 'Castrol',
 'Denny Hamlin': 'Progressive',
 'Ross Chastain': 'Busch Light',
 'Todd Gilliland': "Love's Travel Stops"}

## Pull interest_over_time for each pair

Driver and sponsor go in the *same* `build_payload` call so their 0-100 scores share a
scale — pulling them separately would make the two series incomparable.

In [6]:
TIMEFRAME = '2025-01-01 2025-06-30'  # match your other 2025-6 processed files

results = {}
for driver, sponsor in driver_sponsor_pairs.items():
    df = safe_interest(pytrends, [driver, sponsor], timeframe=TIMEFRAME)
    if df.empty:
        print(f"no data for {driver} / {sponsor}")
        continue
    results[driver] = df.drop(columns='isPartial', errors='ignore')
    time.sleep(1.5)  # be polite between calls

print(len(results))

5


## Does sponsor interest follow driver interest, or the reverse?

For each pair, shift the sponsor series by `k` days/weeks and correlate against the
driver series. If correlation peaks at a *positive* lag (sponsor interest a few days
**after** driver interest), that's evidence driver attention is pulling sponsor
attention along with it, rather than the two moving independently.

In [7]:
def lead_lag_correlation(df, driver_col, sponsor_col, max_lag=5):
    """Correlate driver[t] with sponsor[t+lag] for lag in -max_lag..max_lag.
    Positive lag = sponsor interest trails driver interest by `lag` periods."""
    lags = range(-max_lag, max_lag + 1)
    corrs = {
        lag: df[driver_col].corr(df[sponsor_col].shift(-lag))
        for lag in lags
    }
    return pd.Series(corrs).sort_index()

lead_lag_summary = {}
for driver, df in results.items():
    sponsor_col = [c for c in df.columns if c != driver][0]
    corrs = lead_lag_correlation(df, driver, sponsor_col)
    lead_lag_summary[driver] = corrs

lead_lag_df = pd.DataFrame(lead_lag_summary).T
lead_lag_df['best_lag'] = lead_lag_df.drop(columns='best_lag', errors='ignore').idxmax(axis=1)
lead_lag_df

,-5,-4,-3,-2,-1,0,1,2,3,4,5,best_lag
Austin Hill,-0.109588,-0.063750,0.088062,0.121200,0.068851,0.166813,-0.032240,0.008918,-0.003696,0.062164,0.170737,5
Brad Keselowski,-0.072374,0.059300,0.111144,0.208523,0.080447,-0.081333,-0.021069,0.042061,0.041628,0.090202,0.120283,-2
Denny Hamlin,0.222974,0.179768,0.172523,-0.034260,-0.378874,-0.308347,0.181686,0.224454,0.158078,0.154512,-0.056663,2
Ross Chastain,0.030516,0.050258,0.084846,0.139054,0.147418,0.082096,-0.008086,-0.006441,0.000213,0.049020,0.093494,-1
Todd Gilliland,-0.016927,-0.016830,-0.016734,-0.016638,-0.016544,-0.016451,0.194117,-0.016638,-0.016734,-0.016830,-0.016927,1


In [8]:
# save raw pulls and the lead-lag summary alongside your other processed files
for driver, df in results.items():
    safe_name = driver.replace(' ', '_').replace('.', '')
    df.to_csv(f'data/processed/pytrends_{safe_name}_2025-6.csv')

lead_lag_df.to_csv('data/processed/pytrends_lead_lag_2025-6.csv')